# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">4주차 · 사람이 정한 규칙을 지우고 데이터가 찾은 취향으로 추천하기</mark>

지난 세 주 동안 추천의 규칙은 **사람이 정했습니다.** 주력 섹터로 나누자, 평균 위험도로 나누자 — 강사가 고른 것입니다.

오늘은 **아무도 정하지 않습니다.** 거래 기록만 주고 **모델이 스스로 찾게** 합니다. 그 방법이 **행렬분해**입니다.

---

### 오늘의 구성

| 파트 | 종류 | 어디서 | 하는 일 |
|---|---|---|---|
| 1 | 개념 | 슬라이드 | 추천을 **행렬 완성** 문제로 바꾸고 **R ≈ P·Qᵀ** 로 푼다 |
| 2 | 실습 | **이 노트북 4.2~4.10** | 직접 쪼개고, 점수를 손으로 계산해 보고, 축을 열어 본다 |
| 3 | 마무리 | 슬라이드 | 오늘의 용어 · 점수판 · 다음 주 |

> **지난주에 배운 것은 다시 타이핑하지 않습니다.** 채점은 `recsys.recall_per_user` 한 줄로 부릅니다 — 3주차와 재는 방식이 한 글자도 다르지 않습니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

이 노트북은 **아무것도 설치하지 않고** 브라우저에서 바로 실행할 수 있습니다.

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/04_matrix_factorization.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 "실습 준비" 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다
import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4 · 실습 — 모델에게 취향을 찾게 한다</mark>

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.1 오늘 나오는 말  `[PPT]`</mark>

1. **상호작용 행렬 (interaction matrix)** — 세로가 사람, 가로가 종목인 표. 담은 칸에 **1**, 나머지 칸에 **0** 을 적습니다. 오늘은 285 × 100
2. **행렬 완성 (matrix completion)** — 일부 칸만 아는 표에서 **나머지 칸의 값을 추정**하는 문제. 오늘 푸는 문제입니다
3. **행렬분해 (matrix factorization)** — **R ≈ P·Qᵀ**. 곱하면 원래 표에 가까워지는 두 행렬을 찾는 것. 코드로는 `TruncatedSVD`
4. **중요도 (singular value)** — 축이 **원래 표를 얼마나 설명하는지**. 큰 순서로 나옵니다
5. **잠재 요인 (latent factor)** — 사람과 종목을 설명하는 **축**. 오늘은 8개, 이름은 없습니다
6. **과적합 (overfitting)** — 본 것은 다 맞히는데 **새 것은 못 맞히는** 상태. 4.9 에서 확인합니다

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 · 데이터와 채점 (지난주 그대로)</mark>

**여기서 하는 일** — 데이터를 읽고, 채점에 쓸 것 두 개만 만들어 둡니다.

**채점 함수는 다시 만들지 않습니다.** `recsys.recall_per_user` 가 이미 있습니다 — 3주차에 쓰던 그 방식 그대로(시간순 분할 · 이미 담은 것은 정답에서 빼기 · 맞힐 것이 없으면 채점에서 빼기)입니다.

**잣대가 같아야** 오늘 숫자를 2·3주차 0.2684 와 견줄 수 있습니다.

In [2]:
# 이 셀에서 하는 일 — 오늘 쓸 도구를 불러온다
# 왜 하나 — pandas 로 표를, numpy 로 숫자 묶음을 다루고, recsys.py 에는 1~3주차 함수가 있다
import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구

sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 칸 안쪽 폴더도 찾게 한다

import pandas as pd                         # 표를 다루는 도구. 앞으로 pd 라고 부른다
import numpy as np                          # 숫자 묶음을 빠르게 다루는 도구. np 라고 부른다
import recsys                               # 이 수업용으로 만든 도구 모음

print("도구 준비 완료 · pandas", pd.__version__, "· numpy", np.__version__)   # 버전도 남겨 둔다

도구 준비 완료 · pandas 3.0.5 · numpy 2.5.2


In [3]:
# 이 셀에서 하는 일 — 데이터를 읽고 3주차와 똑같이 학습 구간·채점 구간으로 나눈다
# 왜 하나 — 잣대가 지난주와 같아야 0.2684 와 견줄 수 있다
items, users, interactions = recsys.load()          # 종목·투자자·거래 기록 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 1주차부터 쓰던 시간순 분할
print(f"학습 구간 {len(train):,}건 · 채점 구간 {len(test):,}건 · 자른 날짜 {기준시점.date()}")

이름_사전 = items.set_index("item_id")["name"].to_dict()       # {종목 번호: 이름}
위험도_사전 = items.set_index("item_id")["risk_level"].to_dict()  # {종목 번호: 1~5}

이미_담은것 = train.groupby("user_id")["item_id"].apply(set).to_dict()   # {사람: 학습 구간에 담은 집합}
전체_인기순위 = list(train["item_id"].value_counts().index)              # 많이 담긴 순 (기록 없는 사람용)
print(f"학습 구간에 기록이 있는 사람 {len(이미_담은것)}명 · 인기순위 {len(전체_인기순위)}개")
print("채점은 recsys.recall_per_user 로 합니다 — 3주차와 재는 방식이 같습니다.")

학습 구간 7,095건 · 채점 구간 991건 · 자른 날짜 2026-07-30
학습 구간에 기록이 있는 사람 285명 · 인기순위 100개
채점은 recsys.recall_per_user 로 합니다 — 3주차와 재는 방식이 같습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.2 사람 × 종목 표를 만든다</mark>

**4.2 에서 하는 일** — 거래 기록을 **표 한 장**으로 바꿉니다.

**1. 왜 표로 바꾸나**
- 거래 기록은 「누가 · 무엇을 · 언제」가 한 줄씩 적힌 **긴 목록**입니다. 모델은 이 모양을 못 읽습니다.
- 세로 **사람**, 가로 **종목**인 표로 바꿔 줘야 합니다. 담았으면 **1**, 안 담았으면 **0**.

**2. 이 표의 빈칸이 오늘의 과녁입니다**
- 칸이 285 × 100 = **28,500개**인데 1 이 들어가는 칸은 **7,095개**뿐입니다.
- 이 **0 은 「싫다」가 아니라 「아직 모른다」**입니다. 그 빈칸을 그럴듯한 숫자로 채우는 것이 오늘 할 일입니다.

#### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 새로 나오는 문법 — <code>pd.crosstab</code></mark>

`pd.crosstab(세로, 가로)` — 두 열을 받아 **가로세로 표**를 만듭니다. 칸에는 **그 조합이 몇 번 나왔는지**가 들어갑니다.

`(표 > 0).astype(int)` — 0보다 크면 **1**, 아니면 **0**. `astype` 은 **타입을 바꾼다**는 뜻입니다.
- 우리는 **몇 번 담았는지가 아니라 담았는지 아닌지**만 봅니다.

아래 셀은 마음껏 고쳐 보셔도 됩니다.

In [4]:
# 이 셀에서 하는 일 — 작은 표로 crosstab 을 먼저 해 본다
# 왜 하나 — 8,086건에 쓰기 전에 무슨 일이 일어나는지 5줄로 확인한다
작은기록 = pd.DataFrame({                                # 손으로 만든 거래 기록 5줄
    "사람": ["가", "가", "나", "나", "다"],
    "종목": ["애플", "카카오", "애플", "애플", "네이버"],
})
작은표 = pd.crosstab(작은기록["사람"], 작은기록["종목"])    # 사람 × 종목 표로 펼친다
print("crosstab 으로 펼친 표 — 칸은 담은 횟수")            # 「나」는 애플을 두 번 담았다
print(작은표)

작은표 = (작은표 > 0).astype(int)                         # 담았으면 1, 아니면 0
print("\n1 과 0 으로 누른 표 — 두 번 담았어도 1 입니다")
print(작은표)

crosstab 으로 펼친 표 — 칸은 담은 횟수
종목  네이버  애플  카카오
사람              
가     0   1    1
나     0   2    0
다     1   0    0

1 과 0 으로 누른 표 — 두 번 담았어도 1 입니다
종목  네이버  애플  카카오
사람              
가     0   1    1
나     0   1    0
다     1   0    0


In [5]:
# 이 셀에서 하는 일 — 진짜 데이터로 사람 × 종목 표를 만든다
# 왜 하나 — 이 표가 오늘 쪼갤 대상이다. 학습 구간만 쓴다 — 채점 구간을 보면 반칙이니까
표 = pd.crosstab(train["user_id"], train["item_id"])      # 학습 구간만으로 펼친다
표 = (표 > 0).astype(int)                                  # 담았으면 1, 아니면 0
표 = 표.reindex(columns=items["item_id"], fill_value=0)    # 아무도 안 담은 종목도 열로 세운다

칸_전체 = 표.shape[0] * 표.shape[1]                        # shape = (세로, 가로)
칸_채움 = int(표.values.sum())                             # 1 이 들어간 칸의 개수
print(f"표 크기 — 사람 {표.shape[0]}명 × 종목 {표.shape[1]}개 = 칸 {칸_전체:,}개")
print(f"1 인 칸 {칸_채움:,}개 = 전체의 {칸_채움 / 칸_전체:.1%}   ← 나머지는 전부 0")

print("\n왼쪽 위 귀퉁이만 잘라서 보기 (사람 5명 × 종목 6개)")   # 표가 실제로 어떻게 생겼나
print(표.iloc[:5, :6])

표 크기 — 사람 285명 × 종목 100개 = 칸 28,500개
1 인 칸 7,095개 = 전체의 24.9%   ← 나머지는 전부 0

왼쪽 위 귀퉁이만 잘라서 보기 (사람 5명 × 종목 6개)
item_id  I001  I002  I003  I004  I005  I006
user_id                                    
U0001       0     0     0     0     0     1
U0002       0     1     0     0     0     0
U0003       0     0     0     0     1     0
U0004       1     0     1     1     1     0
U0005       0     0     0     0     0     0


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.3 큰 표를 작은 표 둘로 쪼갠다</mark>

**4.3 에서 하는 일** — 28,500칸짜리 표를 **사람표(285×8)** 와 **종목표(8×100)** 로 쪼갭니다.

**1. 무엇을 기준으로 쪼개나**
- 쪼개면 축이 **종목 수만큼(100개)** 나옵니다. 축마다 **중요도**가 붙는데, 큰 순서로 정렬돼 나옵니다.
- 그중 **특이값이 큰 8개만 남기고 92개는 버립니다.** 이름의 `Truncated`(잘라 쓴다)가 이것입니다.

**2. 무엇을 잃고 무엇을 얻나**
- 버리는 과정에서 **자잘한 취향은 사라지고 큰 취향만** 남습니다.
- 칸이 28,500 → **3,080개**로 줄고, 대신 **빈칸에도 숫자가 생깁니다.**

#### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 새로 나오는 문법 — <code>TruncatedSVD</code></mark>

```python
from sklearn.decomposition import TruncatedSVD
모델 = TruncatedSVD(n_components=8, random_state=42)
사람표 = 모델.fit_transform(표)     # 285 × 8
종목표 = 모델.components_           # 8 × 100
```

- `TruncatedSVD` — 행렬분해(`R ≈ P·Qᵀ`)의 한 방법. **Colab 에 이미 깔려 있어 설치가 없습니다.**
- `n_components=8` — **잠재 요인을 몇 개로 할지**(식의 `k`). 4.9 에서 이 숫자를 바꿔 봅니다.
- `fit_transform` — `fit`(쪼갤 방법을 정한다) + `transform`(그 방법으로 사람표를 만든다)
- `components_` — 같이 나온 **종목표**. 뒤 밑줄은 「학습하고 나서 생긴 값」이라는 표시입니다
- `singular_values_` — 축마다의 **중요도**. `explained_variance_ratio_` 는 그 축이 설명하는 비율입니다

In [6]:
# 이 셀에서 하는 일 — 표를 쪼개고, 축의 중요도와 한 사람의 숫자 8개를 본다
# 왜 하나 — 무엇을 기준으로 쪼갰는지(중요도)와, 사람마다 숫자 8개가 생긴다는 것을 확인한다
from sklearn.decomposition import TruncatedSVD     # 행렬분해 도구

모델 = TruncatedSVD(n_components=8, random_state=42)   # 잠재 요인 8개로 쪼갠다
사람표 = 모델.fit_transform(표)                         # 285 × 8 — 사람마다 숫자 8개
종목표 = 모델.components_                               # 8 × 100 — 종목마다 숫자 8개

print(f"쪼개기 전 — 칸 {표.shape[0] * 표.shape[1]:,}개")
print(f"쪼갠 뒤   — 사람표 {사람표.shape} + 종목표 {종목표.shape} = 칸 {사람표.size + 종목표.size:,}개")

print("\n축마다 붙은 중요도 :", np.round(모델.singular_values_, 1))          # 큰 순서로 나온다
print(f"  8개 축이 원래 표를 설명하는 비율 : {모델.explained_variance_ratio_.sum():.1%}")
print("  축은 100개 나오는데 특이값이 큰 8개만 남긴 것입니다 — 그게 truncated 입니다.")

사람_이름들 = list(표.index)                            # 표의 세로 이름(사람) 목록
열이름 = list(표.columns)                               # 표의 가로 이름(종목 번호) 목록
자리 = 사람_이름들.index("U0003")                       # U0003 이 몇 번째 줄인지
print(f"\nU0003 의 잠재 요인 8개 : {np.round(사람표[자리], 2)}")   # 이름이 없는 숫자 여덟 개

쪼개기 전 — 칸 28,500개
쪼갠 뒤   — 사람표 (285, 8) + 종목표 (8, 100) = 칸 3,080개

축마다 붙은 중요도 : [46.2 21.1 18.4 13.9 13.2 12.8 12.5 11.9]
  8개 축이 원래 표를 설명하는 비율 : 33.9%
  축은 100개 나오는데 특이값이 큰 8개만 남긴 것입니다 — 그게 truncated 입니다.

U0003 의 잠재 요인 8개 : [ 3.16 -1.38  1.43  0.66 -1.09  0.05  0.09  0.28]


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.4 점수 하나가 나오는 법 — 여덟 번 곱해 더한다</mark>

**4.4 에서 하는 일** — 「쪼갠 둘을 다시 곱한다」가 **무슨 계산인지** 숫자로 확인합니다.

- 사람에게 숫자 8개, 종목에게도 숫자 8개가 붙어 있습니다.
- **같은 자리끼리 곱해서 전부 더하면** 그 칸의 점수입니다. 수학에서 **내적**이라고 부릅니다.
- 두 숫자 묶음이 **같은 방향을 가리킬수록** 값이 커집니다. 그래서 「취향이 맞는다」가 큰 점수가 됩니다.

In [7]:
# 이 셀에서 하는 일 — 점수 하나를 손으로 계산해 본다 (U0003 과 TSMC)
# 왜 하나 — 「다시 곱한다」가 여덟 번 곱해 더하는 것이라는 걸 눈으로 확인한다
# U0003 이 아직 안 담은 것 중 점수가 가장 높은 종목을 고른다 (추천 후보)
점수표_미리 = 사람표 @ 종목표                              # 잠깐 곱해서 점수를 본다
종목번호 = None                                           # 고를 종목 번호
for 자리번호 in np.argsort(-점수표_미리[자리]):             # 점수 높은 순으로
    if 표.loc["U0003", 열이름[자리번호]] == 0:              # 아직 안 담은 것이면
        종목번호 = 열이름[자리번호]                          # 그걸 고르고
        break                                             # 멈춘다
종목자리 = 열이름.index(종목번호)                          # 그 종목이 몇 번째 열인지

사람_요인 = 사람표[자리]                                  # U0003 의 숫자 8개
종목_요인 = 종목표[:, 종목자리]                            # 그 종목의 숫자 8개
자리마다_곱 = 사람_요인 * 종목_요인                        # 같은 자리끼리 곱한다

print(f"U0003 의 요인        : {np.round(사람_요인, 2)}")
print(f"{이름_사전[종목번호]} 의 요인 : {np.round(종목_요인, 2)}")
print(f"자리마다 곱하면      : {np.round(자리마다_곱, 2)}")
print(f"\n다 더하면 {자리마다_곱.sum():.3f}  ← 이것이 「U0003 이 이걸 담을 것 같다」는 점수")
print(f"np.dot 으로 한 번에  : {np.dot(사람_요인, 종목_요인):.3f}   ← 같은 계산입니다")

U0003 의 요인        : [ 3.16 -1.38  1.43  0.66 -1.09  0.05  0.09  0.28]
TIGER 미국나스닥100레버리지 의 요인 : [ 0.1  -0.12  0.19 -0.04  0.06 -0.03 -0.02 -0.07]
자리마다 곱하면      : [ 0.32  0.17  0.27 -0.03 -0.07 -0.   -0.   -0.02]

다 더하면 0.641  ← 이것이 「U0003 이 이걸 담을 것 같다」는 점수
np.dot 으로 한 번에  : 0.641   ← 같은 계산입니다


In [8]:
# 이 셀에서 하는 일 — 그 계산을 모든 칸에 한 번에 해서 점수표를 만든다
# 왜 하나 — 사람 285명 × 종목 100개를 하나씩 곱할 수는 없다. @ 하나면 끝난다
점수표 = 사람표 @ 종목표                                 # @ = 행렬 곱하기. 285 × 100 으로 돌아온다
print("점수표 크기 :", 점수표.shape, "  ← 원래 표와 같은 크기")

안_담은것 = []                                           # U0003 이 학습 구간에 안 담은 종목
for 번호 in 열이름:                                      # 종목을 하나씩
    if 표.loc["U0003", 번호] == 0:                       # 표에서 0 이면 안 담은 것
        안_담은것.append(번호)

점수_모음 = {}                                           # {종목 번호: 점수표에 붙은 점수}
for 번호 in 안_담은것:                                    # 안 담은 종목을 하나씩
    점수_모음[번호] = 점수표[자리][열이름.index(번호)]      # 그 칸의 점수를 꺼낸다

print(f"\nU0003 이 안 담은 종목 {len(안_담은것)}개 — 표에서는 전부 0 이었습니다")
print("점수표에는 숫자가 붙어 있습니다 (높은 순 세 개)")
for 번호 in sorted(점수_모음, key=점수_모음.get, reverse=True)[:3]:   # 큰 순으로 셋
    print(f"  {이름_사전[번호]:34} {점수_모음[번호]:.3f}   ← 추천 후보")

점수표 크기 : (285, 100)   ← 원래 표와 같은 크기

U0003 이 안 담은 종목 71개 — 표에서는 전부 0 이었습니다
점수표에는 숫자가 붙어 있습니다 (높은 순 세 개)
  TIGER 미국나스닥100레버리지                 0.641   ← 추천 후보
  Salesforce                         0.605   ← 추천 후보
  iShares Global Clean Energy ETF    0.603   ← 추천 후보


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.5 추천 10개를 고른다  ✏️ 직접 해 보기</mark>

**4.5 에서 하는 일** — 점수표에서 **높은 순으로 10개**를 뽑아 추천 목록을 만듭니다.

**1. 두 가지를 빼고 줍니다**
- **이미 담은 것**은 뺍니다(2주차에 정한 규칙). **학습 구간에 기록이 없는 사람**은 점수표에 줄이 없으니 전체 인기 목록을 줍니다.

**2. 빈칸은 한 줄입니다**
- 점수가 높은 순서대로 **자리 번호**를 얻는 줄입니다. 아래에서 작은 예제로 먼저 연습합니다.

#### <mark style="display:block; background:#bbdefb; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 새로 나오는 문법 — <code>np.argsort</code></mark>

`np.argsort(숫자묶음)` — **작은 순서대로** 그 값이 있던 **자리 번호**를 돌려줍니다. 값이 아니라 **자리**입니다.

- 큰 순서로 받고 싶으면 **앞에 마이너스** — `np.argsort(-숫자묶음)`
- `arg`(자리) + `sort`(정렬) = **「정렬했을 때의 자리 번호」**

In [9]:
# 이 셀에서 하는 일 — 작은 숫자 다섯 개로 argsort 를 먼저 해 본다
# 왜 하나 — 값이 아니라 「자리 번호」가 나온다는 것을 확인하고 넘어간다
점수_다섯개 = np.array([0.1, 0.9, 0.4, 0.7, 0.2])       # 다섯 종목의 점수라고 치자
종목_다섯개 = ["가나다", "라마바", "사아자", "차카타", "파하가"]

print("점수        :", 점수_다섯개)                       # 값 자체
print("작은 순 자리 :", np.argsort(점수_다섯개))           # 가장 작은 0.1 이 0번 자리
print("큰 순 자리   :", np.argsort(-점수_다섯개))          # 마이너스를 붙이면 큰 순

print("\n큰 순으로 이름을 늘어놓으면")
for 자리번호 in np.argsort(-점수_다섯개):                  # 큰 순서대로 자리를 하나씩
    print(f"  {종목_다섯개[자리번호]} ({점수_다섯개[자리번호]})")

점수        : [0.1 0.9 0.4 0.7 0.2]
작은 순 자리 : [0 4 2 3 1]
큰 순 자리   : [1 3 2 4 0]

큰 순으로 이름을 늘어놓으면
  라마바 (0.9)
  차카타 (0.7)
  사아자 (0.4)
  파하가 (0.2)
  가나다 (0.1)


In [10]:
# 이 셀에서 하는 일 — ✏️ 빈칸 1 · 점수가 높은 순으로 추천 10개를 고른다
# 왜 하나 — 여기가 오늘 만드는 추천의 심장이다. 나머지는 지난주 것을 그대로 쓴다
def 추천하기(사람, 이미):
    """그 사람의 점수표 한 줄을 보고 높은 순으로 10개를 돌려준다."""
    if 사람 not in 사람_이름들:                            # 표에 줄이 없는 사람(신규)은
        return recsys.take(전체_인기순위, 이미)[:10]        # 전체 인기 목록으로 준다

    내_점수 = 점수표[사람_이름들.index(사람)]               # 그 사람 줄 = 종목 100개의 점수
    내_순서 = []                                          # 점수가 높은 종목부터 담을 목록
    for 자리번호 in np.argsort(-내_점수):                  # ← ✏️ 빈칸 : 높은 순 자리 번호
        내_순서.append(열이름[자리번호])                    # 자리 번호를 종목 번호로 바꿔 담는다
    return recsys.take(내_순서, 이미)[:10]                 # 이미 담은 것을 빼고 위에서 10개


내_추천 = 추천하기("U0003", 이미_담은것["U0003"])           # 만들었으면 한 명 돌려 본다
print("U0003 에게 줄 추천 10개")
for 순위, 번호 in enumerate(내_추천, start=1):             # enumerate = 번호를 붙여 준다
    print(f"  {순위:2}위  {이름_사전[번호]:34} 위험도 {위험도_사전[번호]}")

U0003 에게 줄 추천 10개
   1위  TIGER 미국나스닥100레버리지                 위험도 5
   2위  Salesforce                         위험도 4
   3위  iShares Global Clean Energy ETF    위험도 4
   4위  ProShares UltraPro Short QQQ       위험도 5
   5위  Vanguard FTSE Developed Markets ETF 위험도 3
   6위  LG에너지솔루션                           위험도 5
   7위  Global X Lithium & Battery Tech ETF 위험도 5
   8위  Vanguard S&P 500 ETF               위험도 3
   9위  TIGER 200                          위험도 3
  10위  알테오젠                               위험도 5


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.6 채점한다 — 지난주 자를 그대로 쓴다</mark>

**4.6 에서 하는 일** — 방금 만든 추천 방식을 **3주차와 같은 자**로 잽니다.

- 채점 코드를 다시 쓰지 않습니다. **`recsys.recall_per_user` 한 줄**입니다.
- 재는 방법이 한 글자도 안 바뀌었으니, 숫자가 달라졌다면 원인은 **추천 방식** 하나뿐입니다.
- 점수 말고 하나 더 봅니다 — **서로 다른 목록이 몇 가지** 나왔는지. 2주차에는 15가지뿐이었습니다.

In [11]:
# 이 셀에서 하는 일 — 3주차와 같은 자로 채점한다 (한 줄)
# 왜 하나 — 잣대를 다시 만들지 않는다. 같은 함수를 부르는 것이 「같은 자」라는 증거다
점수들 = recsys.recall_per_user(추천하기, train, test)     # {사람: Recall@10} — 3주차 방식 그대로
행렬분해_점수 = float(np.mean(list(점수들.values())))       # 점수판에 적을 값

print(f"4주차 행렬분해 — Recall@10 = {행렬분해_점수:.4f}   (채점 대상 {len(점수들)}명)")
print(f"2·3주차 세그먼트 — Recall@10 = 0.2684")             # 견줄 상대
print(f"\n→ {(행렬분해_점수 - 0.2684) / 0.2684:+.1%} 올랐습니다. 바뀐 것은 추천을 만드는 방법 하나입니다.")

4주차 행렬분해 — Recall@10 = 0.3918   (채점 대상 266명)
2·3주차 세그먼트 — Recall@10 = 0.2684

→ +46.0% 올랐습니다. 바뀐 것은 추천을 만드는 방법 하나입니다.


In [12]:
# 이 셀에서 하는 일 — 서로 다른 추천 목록이 몇 가지 나왔는지 센다
# 왜 하나 — 2주차에는 15가지뿐이었다. 개인화가 실제로 되는지는 이 숫자가 말해 준다
목록_모음 = set()                                          # 중복을 저절로 없애 주는 묶음
for 사람 in 점수들:                                        # 채점한 사람을 하나씩
    목록_모음.add(tuple(추천하기(사람, 이미_담은것.get(사람, set()))))   # 목록을 통째로 넣는다

print(f"채점 대상 {len(점수들)}명에게 나간 목록 — 서로 다른 것이 {len(목록_모음)}가지")
print("2주차에는 세그먼트가 15개였으니 15가지였습니다.")
print("\n→ 같은 세그먼트면 같은 목록을 받던 문제가 풀렸습니다.")

채점 대상 266명에게 나간 목록 — 서로 다른 것이 252가지
2주차에는 세그먼트가 15개였으니 15가지였습니다.

→ 같은 세그먼트면 같은 목록을 받던 문제가 풀렸습니다.


**결과 — 규칙을 안 정했더니 올랐습니다**

| 주차 | 규칙을 누가 정했나 | Recall@10 |
|---|---|---|
| 1주차 | 규칙이랄 것도 없음 (인기순) | 0.2163 |
| 2·3주차 | 사람 — 주력 섹터·평균 위험도 | 0.2684 |
| **4주차** | **아무도 안 정함 — 모델이 찾음** | **0.3918** |

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.7 모델이 찾아낸 축을 열어 본다</mark>

**4.7 에서 하는 일** — 점수가 올랐으니 **모델이 무엇을 찾은 것인지** 들여다봅니다. 세 가지를 확인합니다.

1. **1번 축의 정체** — 중요도가 유독 컸던 그 축은 취향일까요, 아닐까요
2. **2번 축의 정체** — 이름이 없는데도 뜻이 있는지
3. **빈칸이 채워지는 이유** — 안 담은 칸에 점수가 붙은 근거가 무엇인지

In [13]:
# 이 셀에서 하는 일 — 1번 축이 무엇을 잡고 있는지 확인한다
# 왜 하나 — 중요도가 46.2 로 유독 컸다. 그 이유를 알면 나머지 축을 읽는 눈이 생긴다
축1_큰쪽 = []                                             # 1번 축 값이 큰 종목 이름
for 자리번호 in np.argsort(-종목표[0])[:3]:                # 종목표 첫 줄 = 1번 축
    축1_큰쪽.append(이름_사전[열이름[자리번호]])
많이_담긴것 = []                                          # 학습 구간에서 많이 담긴 종목
for 번호 in 전체_인기순위[:3]:                             # 인기순위 위에서 셋
    많이_담긴것.append(이름_사전[번호])

print("1번 축 값이 큰 종목 3개 :", 축1_큰쪽)
print("전체에서 많이 담긴 3개   :", 많이_담긴것)           # 두 줄을 견줘 본다

담은_개수 = 표.values.sum(axis=1)                         # 사람마다 몇 개 담았나
상관 = np.corrcoef(사람표[:, 0], 담은_개수)[0, 1]          # 1번 축 값과 담은 개수의 상관
print(f"\n사람표 1번 축이 모두 양수인가 : {bool((사람표[:, 0] > 0).all())}")
print(f"1번 축 값과 담은 개수의 상관    : {상관:.3f}   ← 1 에 가까우면 거의 같이 움직인다")
print("\n→ 1번 축은 취향이 아니라 「얼마나 많이 담았나」입니다. 취향은 2번 축부터 갈립니다.")

1번 축 값이 큰 종목 3개 : ['iShares Core MSCI EAFE ETF', 'Vanguard FTSE Developed Markets ETF', 'TIGER 미국S&P500']
전체에서 많이 담긴 3개   : ['iShares Core MSCI EAFE ETF', 'Vanguard FTSE Developed Markets ETF', 'TIGER 미국S&P500']

사람표 1번 축이 모두 양수인가 : True
1번 축 값과 담은 개수의 상관    : 0.955   ← 1 에 가까우면 거의 같이 움직인다

→ 1번 축은 취향이 아니라 「얼마나 많이 담았나」입니다. 취향은 2번 축부터 갈립니다.


In [14]:
# 이 셀에서 하는 일 — 2번 축의 양쪽 끝에 어떤 종목이 있는지 본다
# 왜 하나 — 이름 없는 축에 뜻이 있다는 것을 눈으로 확인한다
def 축의_끝(자리번호들):                                   # 자리 번호 묶음 → 이름과 평균 위험도
    이름들 = []                                           # 종목 이름을 담을 목록
    위험_합 = 0                                           # 위험도를 더해 갈 그릇
    for 자리번호 in 자리번호들:                             # 자리 번호를 하나씩
        번호 = 열이름[자리번호]                             # 자리 번호 → 종목 번호
        이름들.append(이름_사전[번호])                       # 이름을 담고
        위험_합 = 위험_합 + 위험도_사전[번호]                # 위험도를 더한다
    return 이름들, 위험_합 / len(자리번호들)                # 이름 목록과 평균 위험도

큰쪽_이름, 큰쪽_위험 = 축의_끝(np.argsort(-종목표[1])[:5])    # 2번 축 값이 큰 5개
작은쪽_이름, 작은쪽_위험 = 축의_끝(np.argsort(종목표[1])[:5])  # 2번 축 값이 작은 5개

print("2번 축의 큰 쪽   :", 큰쪽_이름[:3], f"· 평균 위험도 {큰쪽_위험:.1f}")
print("2번 축의 작은 쪽 :", 작은쪽_이름[:3], f"· 평균 위험도 {작은쪽_위험:.1f}")
print("\n→ 2번 축은 「안전이냐 위험이냐」였습니다. 위험도를 알려 준 적이 없는데 찾아냈습니다.")

2번 축의 큰 쪽   : ['iShares 20+ Year Treasury Bond ETF', 'ACE 미국배당다우존스', 'SK텔레콤'] · 평균 위험도 2.0
2번 축의 작은 쪽 : ['KODEX 코스닥150레버리지', 'Invesco QQQ Trust', 'TIGER 미국나스닥100레버리지'] · 평균 위험도 4.4

→ 2번 축은 「안전이냐 위험이냐」였습니다. 위험도를 알려 준 적이 없는데 찾아냈습니다.


In [15]:
# 이 셀에서 하는 일 — 빈칸에 점수가 붙은 근거를 따라가 본다
# 왜 하나 — 「마법으로 채워진다」가 아니라 「이웃이 담았기 때문」이라는 것을 확인한다
길이 = np.sqrt((사람표 * 사람표).sum(axis=1))             # 사람마다 숫자 8개의 길이
방향 = 사람표 / 길이[:, None]                             # 길이를 1로 맞춘다 — 방향만 남는다
닮음표 = 방향 @ 방향.T                                    # 285 × 285. 1 에 가까울수록 닮았다

담은것 = {}                                              # {사람: 학습 구간에 담은 집합}
for 사람, 그_사람의_기록 in train.groupby("user_id"):      # 사람별로 묶어서
    담은것[사람] = set(그_사람의_기록["item_id"])           # 담은 종목을 집합으로

이웃들 = []                                              # U0003 과 요인이 가까운 5명
for 이웃_자리 in np.argsort(-닮음표[자리])[1:6]:           # 0번은 자기 자신이라 1번부터
    이웃들.append(사람_이름들[이웃_자리])

담은_이웃 = 0                                            # 그 5명 중 TSMC 를 담은 사람 수
for 이웃 in 이웃들:                                       # 이웃을 하나씩
    if 종목번호 in 담은것[이웃]:                           # 그 종목을 담았으면
        담은_이웃 = 담은_이웃 + 1                          # 센다

print(f"U0003 의 점수 1위 「{이름_사전[종목번호]}」 — 본인은 안 담았습니다")
print(f"  요인이 가까운 5명 {이웃들}")
print(f"  그중 {담은_이웃}명이 담았습니다   ← 빈칸은 이웃의 패턴으로 메워집니다")

# 한 사람만 보면 우연일 수 있으니 285명 전부로 세어 본다 (슬라이드의 13.5 vs 7.1)
가까운_합 = 0                                            # 가까운 3명과 겹친 종목 수
무작위_합 = 0                                            # 아무 3명과 겹친 종목 수
셈 = 0                                                   # 몇 쌍을 셌는지
주사위 = np.random.default_rng(0)                         # 아무나 고를 때 쓸 난수. 0 은 고정용
for 내_번호, 나 in enumerate(사람_이름들):                 # 285명을 한 명씩
    for 이웃_자리 in np.argsort(-닮음표[내_번호])[1:4]:     # 가장 닮은 세 명
        가까운_합 = 가까운_합 + len(담은것[나] & 담은것[사람_이름들[이웃_자리]])
        셈 = 셈 + 1
    뽑은_수 = 0                                          # 아무나 세 명을 뽑을 때까지
    while 뽑은_수 < 3:
        아무_자리 = int(주사위.integers(len(사람_이름들)))  # 아무 자리나 하나
        if 아무_자리 == 내_번호:                           # 자기 자신이면 다시
            continue
        무작위_합 = 무작위_합 + len(담은것[나] & 담은것[사람_이름들[아무_자리]])
        뽑은_수 = 뽑은_수 + 1

print(f"\n요인이 가까운 3명과 겹치는 종목  평균 {가까운_합 / 셈:.1f}개")
print(f"아무나 고른 3명과 겹치는 종목    평균 {무작위_합 / 셈:.1f}개   ← 두 배 가까이 차이")

복원 = 점수표[표.values == 1].mean(), 점수표[표.values == 0].mean()   # 1이던 칸 · 0이던 칸
print(f"\n원래 1이던 칸의 복원값 평균 {복원[0]:.2f} · 0이던 칸 {복원[1]:.2f}")
print("→ 1 이 1 로 돌아오지 않습니다. 복원이 아니라 「그럴듯하게 메우기」입니다.")

U0003 의 점수 1위 「TIGER 미국나스닥100레버리지」 — 본인은 안 담았습니다
  요인이 가까운 5명 ['U0063', 'U0243', 'U0090', 'U0231', 'U0171']
  그중 4명이 담았습니다   ← 빈칸은 이웃의 패턴으로 메워집니다

요인이 가까운 3명과 겹치는 종목  평균 13.5개
아무나 고른 3명과 겹치는 종목    평균 7.1개   ← 두 배 가까이 차이

원래 1이던 칸의 복원값 평균 0.53 · 0이던 칸 0.16
→ 1 이 1 로 돌아오지 않습니다. 복원이 아니라 「그럴듯하게 메우기」입니다.


**결과 — 아무도 이름을 안 붙여 준 축에 뜻이 있습니다**

- **1번 축**은 취향이 아니라 **인기**였습니다. 값이 큰 종목이 전체 인기 종목과 같고, 사람 쪽은 담은 개수와 거의 같이 움직입니다.
- **2번 축**은 한쪽 끝에 국채·배당·통신, 반대쪽에 레버리지가 모였습니다. **안전 ↔ 위험** 축입니다. 우리가 준 것은 **누가 무엇을 담았는지**뿐인데 행동만 보고 찾아냈습니다.
- **빈칸**은 마법이 아니라 **요인이 가까운 사람들이 담은 것**으로 메워집니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.8 이 방법이 못 하는 것</mark>

**4.8 에서 하는 일** — 좋은 점만 배우면 반쪽입니다. **어디까지만 되는지**를 숫자로 확인합니다.

- 안 담은 칸을 **0(싫다)으로 놓고** 계산합니다. 실제로는 「아직 모른다」인데요.
- 8개로 줄였으니 **못 살리는 것**이 생깁니다. 얼마나 되는지 세어 봅니다.
- 그래서 **ALS · BPR** 같은 방법이 나왔습니다. 면접에서 이어지는 질문이 정확히 여기입니다.

In [16]:
# 이 셀에서 하는 일 — 8개로 줄이면서 잃은 것이 얼마나 되는지 센다
# 왜 하나 — 한계를 숫자로 알아야 다음 방법(ALS·BPR)이 왜 나왔는지 이해된다
원래_1인칸 = 점수표[표.values == 1]                        # 원래 담았던 칸의 복원값들
못_살린것 = (원래_1인칸 < 0.3).sum()                       # 0.3 에도 못 미치는 칸의 개수

print(f"원래 1이던 칸 {len(원래_1인칸):,}개 중")
print(f"  복원값이 0.3 에도 못 미치는 칸 {못_살린것:,}개 = {못_살린것 / len(원래_1인칸):.0%}")
print("\n→ 축 8개로 줄였으니 못 살리는 것이 생깁니다. 이것이 근사의 대가입니다.")
print("→ 게다가 안 담은 칸을 「싫다」로 놓고 계산합니다. 실제로는 「아직 모른다」인데요.")
print("→ ALS · BPR 은 안 담은 칸을 덜 믿는 쪽으로 계산을 고친 방법들입니다.")

원래 1이던 칸 7,095개 중
  복원값이 0.3 에도 못 미치는 칸 1,499개 = 21%

→ 축 8개로 줄였으니 못 살리는 것이 생깁니다. 이것이 근사의 대가입니다.
→ 게다가 안 담은 칸을 「싫다」로 놓고 계산합니다. 실제로는 「아직 모른다」인데요.
→ ALS · BPR 은 안 담은 칸을 덜 믿는 쪽으로 계산을 고친 방법들입니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.9 요인 개수를 바꿔 본다  ✏️ 직접 해 보기</mark>

**4.9 에서 하는 일** — 잠재 요인 개수만 바꿔 **다섯 번** 돌려 보고 점수를 견줍니다.

- 8개는 **정답이 아니라 골라 본 값**입니다. 많이 주면 좋아질 것 같지만 그렇지 않습니다.
- **설명 비율**도 같이 봅니다. 축이 많아지면 원래 표는 더 잘 설명하는데 **점수는 어떻게 되는지** 보세요.
- 마지막에 **몇 개로 할지 정하고 이유를 한 줄** 적습니다. 오늘의 과제입니다.

In [17]:
# 이 셀에서 하는 일 — ✏️ 빈칸 2 · 요인 개수만 바꿔 다섯 번 돌린다
# 왜 하나 — 「많을수록 좋다」가 아니라는 것을 점수로 확인한다
해볼_개수들 = [2, 4, 8, 16, 32]        # ← ✏️ 빈칸 : 돌려 볼 요인 개수를 넣어 보세요

print("요인 개수   설명 비율   Recall@10")
결과_모음 = {}                                              # {요인 개수: 점수}
for 개수 in 해볼_개수들:                                    # 개수를 하나씩 바꿔 가며
    그때_모델 = TruncatedSVD(n_components=개수, random_state=42)   # 그 개수로 쪼갠다
    그때_사람표 = 그때_모델.fit_transform(표)                # 285 × 개수
    그때_점수표 = 그때_사람표 @ 그때_모델.components_         # 다시 곱해 점수표

    def 그때_추천하기(사람, 이미, 점수표=그때_점수표):        # 위 추천 함수와 같은 방식
        if 사람 not in 사람_이름들:
            return recsys.take(전체_인기순위, 이미)[:10]
        내_순서 = []
        for 자리번호 in np.argsort(-점수표[사람_이름들.index(사람)]):
            내_순서.append(열이름[자리번호])
        return recsys.take(내_순서, 이미)[:10]

    그때_점수들 = recsys.recall_per_user(그때_추천하기, train, test)   # 같은 자로 잰다
    결과_모음[개수] = float(np.mean(list(그때_점수들.values())))
    설명비율 = 그때_모델.explained_variance_ratio_.sum()       # 원래 표를 얼마나 설명하나
    print(f"  {개수:>5}개     {설명비율:>5.1%}     {결과_모음[개수]:.4f}")

가장_높은_개수 = max(결과_모음, key=결과_모음.get)           # 점수가 가장 높은 개수
print(f"\n→ 가장 높은 것은 요인 {가장_높은_개수}개 · {결과_모음[가장_높은_개수]:.4f} 입니다.")
print("→ 설명 비율은 계속 오르는데 점수는 떨어집니다. 표를 더 잘 외운 것이지 추천을 더 잘한 게 아닙니다.")

요인 개수   설명 비율   Recall@10
      2개     10.7%     0.2760
      4개     21.3%     0.3452
      8개     33.9%     0.3918


     16개     49.3%     0.3475


     32개     68.2%     0.2985

→ 가장 높은 것은 요인 8개 · 0.3918 입니다.
→ 설명 비율은 계속 오르는데 점수는 떨어집니다. 표를 더 잘 외운 것이지 추천을 더 잘한 게 아닙니다.


In [18]:
# 이 셀에서 하는 일 — ✏️ 내가 고른 요인 개수와 그 이유를 적는다
# 왜 하나 — 오늘의 과제는 점수가 아니라 「왜 그렇게 정했는지 말할 수 있는 것」이다
내가_고른_개수 = 8                      # ← ✏️ 바꿔도 됩니다
내_이유 = "점수가 가장 높았고, 더 늘리면 외우기 시작해서"   # ← ✏️ 한 줄로 적어 주세요

print(f"내가 고른 요인 개수 : {내가_고른_개수}개")           # 위에서 적은 값을 확인
print(f"고른 이유           : {내_이유}")
if 내가_고른_개수 in 결과_모음:                              # 4.9 에서 돌려 본 개수라면
    print(f"그때의 점수         : {결과_모음[내가_고른_개수]:.4f}")
else:                                                       # 안 돌려 본 개수라면
    print("그때의 점수         : 위 목록에 넣고 다시 돌려 보세요.")

내가 고른 요인 개수 : 8개
고른 이유           : 점수가 가장 높았고, 더 늘리면 외우기 시작해서
그때의 점수         : 0.3918


**결과 — 산 모양이 나옵니다**

요인을 **네 배로 늘렸는데 점수는 떨어졌습니다.** 이것이 **과적합**입니다.

축이 많아지면 그 사람이 한 번 담아 본 것까지 전부 외울 수 있게 됩니다. 그런데 **외운 것으로는 안 본 것을 못 맞힙니다.**

데이터가 바뀌면 가장 높은 자리도 옮겨 갑니다. 그래서 **매번 돌려 보고 정합니다.**

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.10 기존과 신규를 갈라서 본다</mark>

**4.10 에서 하는 일** — 3주차와 똑같이 **기존**과 **신규**로 갈라 평균을 따로 냅니다.

- 3주차에 신규 15명이 **0.1653** 이었습니다. 모델을 썼으니 좋아졌을까요.
- 새로 채점하지 않습니다. 4.6 에서 받아 둔 **사람별 점수 266개를 두 바구니에 나눠 담고** 평균만 따로 냅니다.

In [19]:
# 이 셀에서 하는 일 — 채점 대상을 기존·신규로 갈라 평균을 따로 낸다
# 왜 하나 — 전체 평균 하나가 무엇을 가리고 있는지 보려는 것이다
신규_투자자 = set(test["user_id"]) - set(train["user_id"])   # 빼기(-) = 채점에만 있는 사람
기존_점수들 = {}                                            # {기존 투자자: 점수}
신규_점수들 = {}                                            # {신규 투자자: 점수}
for 사람, 점수 in 점수들.items():                            # 4.6 에서 받아 둔 점수를 하나씩
    if 사람 in 신규_투자자:                                  # 신규 명단에 있으면
        신규_점수들[사람] = 점수                              # 신규 바구니에
    else:                                                    # 아니면
        기존_점수들[사람] = 점수                              # 기존 바구니에

평균 = lambda 점수모음: float(np.mean(list(점수모음.values())))   # 바구니의 평균을 내는 짧은 함수
print(f"전체 {len(점수들)}명  {평균(점수들):.4f}   (3주차 0.2684)")      # 괄호 안이 3주차 값
print(f"기존 {len(기존_점수들)}명  {평균(기존_점수들):.4f}   (3주차 0.2746)")
print(f"신규  {len(신규_점수들)}명  {평균(신규_점수들):.4f}   (3주차 0.1653)")
print("\n→ 신규는 표에 줄 자체가 없어 전체 인기 목록을 받습니다. 지난주와 같은 점수입니다.")

전체 266명  0.3918   (3주차 0.2684)
기존 251명  0.4054   (3주차 0.2746)
신규  15명  0.1653   (3주차 0.1653)

→ 신규는 표에 줄 자체가 없어 전체 인기 목록을 받습니다. 지난주와 같은 점수입니다.


**결과 — 모델이 못 푸는 문제가 있습니다**

기존 투자자는 크게 올랐는데 **신규 15명은 3주차와 소수점 끝자리까지 같습니다.**

학습 구간 기록이 0건이라 **사람 × 종목 표에 줄 자체가 없습니다.** 줄이 없으니 요인 8개도 안 나오고, 모델이 점수를 매길 방법이 없습니다.

**모델이 좋아진다고 모든 문제가 풀리지는 않습니다.** 금융 앱이 가입 직후에 투자 성향 설문을 받는 이유가 이것입니다 — **재료를 만들어 내는 것**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">4.11 오늘의 마무리  `[PPT]`</mark>

1. **0.2684 → 0.3918** — 10주 중 가장 많이 오른 주입니다.
2. **목록이 15가지에서 252가지로** — 2주차에 남겨 둔 숙제가 풀렸습니다.
3. **신규 15명은 그대로** — 재료가 없으면 모델도 못 합니다.

다음 주에는 종목을 **전부 재지 않고** 후보를 먼저 추립니다. 점수가 아니라 **규모가 커져도 버티는 구조**를 얻는 주입니다.

In [20]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 4주차는 10주 중 가장 크게 오른 줄이다. 다음 주부터는 이 값과 견준다
recsys.record(4, "행렬분해 SVD · 요인 8개", 행렬분해_점수,
              note="규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다")

print("점수판에 4주차를 기록했습니다.\n")   # \n = 한 줄 띄우기
recsys.leaderboard(upto=4)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 4 · 행렬분해 SVD · 요인 8개 · Recall@10 = 0.3918
점수판에 4주차를 기록했습니다.



,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
